# Exploratory Data Analysis for the Australian Weather Dataset
For this assignment, the exploratory data analysis is split from the rest of the tasks to make the code less connected. This is done after experience gained from the previous assignment that the EDA would be shuffled around the notebook depending on which other aspects of the task needed to be done in which order. This also makes it easier to work on the project in parallel and ensure that once the EDA is finished, it does not have to tested after every change in the main notebook. In summary, the EDA for the Australian Weather Dataset is decoupled from the main notebook to get the advantages of loosely coupled code.



In [ ]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import TargetEncoder, OrdinalEncoder

In [ ]:
eda_dataset = pd.read_csv('Dataset/weatherAUS.csv')
eda_dataset.head()

In [ ]:
eda_dataset.info()

In [ ]:
eda_dataset.isna().sum()

Rows with missing values in the target value are dropped since they offer no value to supervised learning methods.

In [ ]:
eda_dataset = eda_dataset.dropna(subset=["RainTomorrow"])

eda_dataset.isna().sum()

Making a numerical dataset and a categorical dataset to make analysis easier

## Temporary Encoding
For certain parts of the analysis, such as the correlation heatmap, it is necessary to use encoded data. There is therefore a temporary encoding of the categorical and time series values connected to the dataset. The values are encoded in such a way that all parts of features are carried forward in the dataset, and that the encoding makes sense for the feature domain. For example, ordinal encoder is not used on location, since there is no inherent order (other than perhaps coordinate location) to them. Another example is that it may be that the year is unnecessary as a feature since historically rain is hard to predict based on yearly intervals. But, it is still included since this is the analysis part.


In [ ]:
# Create a new encoded version of the dataset which will be used for certain parts of the analysis.
encoded_eda_dataset = eda_dataset.copy()

In [ ]:
encoded_eda_dataset['RainTomorrow'] = encoded_eda_dataset['RainTomorrow'].replace({'No':0, 'Yes':1}).astype(float)

Encoding of categorical data

In [ ]:
# Target encoding of location
x = encoded_eda_dataset[['Location']]
y = encoded_eda_dataset['RainTomorrow']

encoder = TargetEncoder()
encoded_eda_dataset['Location'] = encoder.fit_transform(x,y)

Encoding of numerical data

In [ ]:
# Binary Encoding of RainToday
encoded_eda_dataset['RainToday'] = encoded_eda_dataset['RainToday'].replace({'No': 0, 'Yes': 1}).astype(float)

### Cyclical encoding

This dataset includes multiple features which have cyclical properties. This means that the order of the data is important, but there is a wrap-around of the order at the start and finish. The date includes seasonality, with certain patterns reoccurring depending on which time of year it is. For example, in Australia, rain is more common in the summer. The wind direction is also cyclical, since the direction loops around for every full loop. Ordinal encoding would not retain the wrap-around property that cyclical features contain. Cyclical encoding is used for these features instead.

The relevant features with cyclic properties are as follows:
- WindGustDir
- WindDir9am
- WindDir3am
- date: day
- date: month

A general function for encoding with cyclic data is created which takes a Pandas series object and its corresponding period length. The function returns the same pandas series, encoded for the given period. This encoding includes both a cosine and sine wave fitted version of the Pandas series. Both are needed to keep track of the cycle, since multiple values share the same sine and cosine values in the same cycle. The function only works for numerical data.

In [ ]:
# Function that takes a Pandas series and a corresponding period.
# Only pass numerical series.
# def cyclicEncoder(series, period):
#     # Filter for missing values. (Not needed for main notebook.)
#     # To prevent missing values from ruining the encoding, a check is made with the following requirement:
#     missing = series.isna()
#     # If values are missing, they are simply passed through.
#     sin = np.where(missing, np.nan, np.sin(2*np.pi*series/period))
#     cos = np.where(missing, np.nan, np.cos(2*np.pi*series/period))
#     return sin, cos

def cyclicEncoder(series, period):

    sin = np.sin(2*np.pi*series/period)
    cos = np.cos(2*np.pi*series/period)
    return sin, cos



#### Data Preparation

In [ ]:
# Ready date related data
# Convert to Pandas datetime format
encoded_eda_dataset["Date"] = pd.to_datetime(encoded_eda_dataset["Date"])
# Extract year, month and day based on year.
# Year is already "pre encoded" and can be left untouched.
encoded_eda_dataset["Year"] = encoded_eda_dataset["Date"].dt.year
encoded_eda_dataset["Month"] = encoded_eda_dataset["Date"].dt.month
encoded_eda_dataset["DayOfYear"] = encoded_eda_dataset["Date"].dt.day_of_year

#For analysis, there is a benefit of separating the Date feature as well.
#Therefore, it is done to the unencoded dataset as well.

eda_dataset["Date"] = pd.to_datetime(eda_dataset["Date"])
eda_dataset["Month"] = eda_dataset["Date"].dt.month
eda_dataset["DayOfYear"] = eda_dataset["Date"].dt.day_of_year
eda_dataset["Year"] = eda_dataset["Date"].dt.year

In [ ]:
# Ready wind related data
# Find the unique entries in the wind columns
encoded_eda_dataset["WindGustDir"].unique()

For analytical purposes, the missing values are kept. The rest of the values are manually encoded using domain knowledge.

In [ ]:
encoded_eda_dataset['WindGustDir'] = encoded_eda_dataset['WindGustDir'].replace({'N': 16,
                                                                                 'NNE': 1,
                                                                                 'NE': 2,
                                                                                 'ENE': 3,
                                                                                 'E': 4,
                                                                                 'ESE': 5,
                                                                                 'SE': 6,
                                                                                 'SSE': 7,
                                                                                 'S': 8,
                                                                                 'SSW': 9,
                                                                                 'SW': 10,
                                                                                 'WSW': 11,
                                                                                 'W': 12,
                                                                                 'WNW': 13,
                                                                                 'NW': 14,
                                                                                 'NNW': 15,
                                                                                 }).astype(float)
# Floats has to be used for the nan objects to be translated to floats as well.
# Otherwise, the array with nans will be of dtype object which will cause the sin and cos
# functions from numpy to fail.

encoded_eda_dataset['WindDir9am'] = encoded_eda_dataset['WindDir9am'].replace({'N': 0,
                                                                                 'NNE': 1,
                                                                                 'NE': 2,
                                                                                 'ENE': 3,
                                                                                 'E': 4,
                                                                                 'ESE': 5,
                                                                                 'SE': 6,
                                                                                 'SSE': 7,
                                                                                 'S': 8,
                                                                                 'SSW': 9,
                                                                                 'SW': 10,
                                                                                 'WSW': 11,
                                                                                 'W': 12,
                                                                                 'WNW': 13,
                                                                                 'NW': 14,
                                                                                 'NNW': 15,
                                                                                 }).astype(float)

encoded_eda_dataset['WindDir3pm'] = encoded_eda_dataset['WindDir3pm'].replace({'N': 0,
                                                                                 'NNE': 1,
                                                                                 'NE': 2,
                                                                                 'ENE': 3,
                                                                                 'E': 4,
                                                                                 'ESE': 5,
                                                                                 'SE': 6,
                                                                                 'SSE': 7,
                                                                                 'S': 8,
                                                                                 'SSW': 9,
                                                                                 'SW': 10,
                                                                                 'WSW': 11,
                                                                                 'W': 12,
                                                                                 'WNW': 13,
                                                                                 'NW': 14,
                                                                                 'NNW': 15,
                                                                                 }).astype(float)
# Check for array datatype and encoded values.
encoded_eda_dataset['WindGustDir'].unique()

#### Use the Cyclic Encoding Function

In [ ]:
# For future use
# encoded_eda_dataset["Month_Sin"],encoded_eda_dataset["Month_Cos"]  = cyclicEncoder(encoded_eda_dataset["Month"], MONTH_PERIOD)
# encoded_eda_dataset["Day_Sin"],encoded_eda_dataset["Day_Cos"]  = cyclicEncoder(encoded_eda_dataset["DayOfYear"], encoded_eda_dataset['Days_in_year'])
# encoded_eda_dataset["WindGustDir_Sin"],encoded_eda_dataset["WindGustDir_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindGustDir"], WIND_DIR)
# encoded_eda_dataset["WindDir9am_Sin"],encoded_eda_dataset["WindDir9am_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindDir9am"], WIND_DIR)
# encoded_eda_dataset["WindDir3pm_Sin"],encoded_eda_dataset["WindDir3pm_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindDir3pm"], WIND_DIR)

In [ ]:
# Period lengths:
MONTH_PERIOD = 12
WIND_DIR = 16
# The day is more complicated since it varies based on the year.
# A new temporary pandas series is created and passed in the period based on the year
year = encoded_eda_dataset['Year']
isLeapYear = ((year % 4 == 0) & (year % 100 != 0)) | (year % 400 == 0)
encoded_eda_dataset['DaysInYear'] = np.select([isLeapYear], [366], default=365)


encoded_eda_dataset["Month_Sin"],encoded_eda_dataset["Month_Cos"]  = cyclicEncoder(encoded_eda_dataset["Month"], MONTH_PERIOD)
encoded_eda_dataset["Day_Sin"],encoded_eda_dataset["Day_Cos"]  = cyclicEncoder(encoded_eda_dataset["DayOfYear"], encoded_eda_dataset['DaysInYear'])
encoded_eda_dataset["WindGustDir_Sin"],encoded_eda_dataset["WindGustDir_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindGustDir"], WIND_DIR)
encoded_eda_dataset["WindDir9am_Sin"],encoded_eda_dataset["WindDir9am_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindDir9am"], WIND_DIR)
encoded_eda_dataset["WindDir3pm_Sin"],encoded_eda_dataset["WindDir3pm_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindDir3pm"], WIND_DIR)

# Remove redundant columns
encoded_eda_dataset.drop(["Date", "DaysInYear", "Month", "DayOfYear", "WindGustDir", "WindDir9am", "WindDir3pm"], axis=1, inplace=True)

In [ ]:
encoded_eda_dataset["Month_Sin"].unique()
# encoded_eda_dataset["Month_Cos"].unique()
# encoded_eda_dataset["WindGustDir_Sin"].unique()

### Check encoding

In [ ]:
encoded_eda_dataset.head()

## Data Analysis

### Univariate Analysis

In [ ]:
# See how many different locations are used.
eda_dataset["Location"].nunique()

In [ ]:
# Check if every location has the same amount of observations
eda_dataset.value_counts("Location")

As can be seen from the result, the gap between the location with the most and fourth least entries is small, but not negligible. The gap is 3436 - 3001 = 435, which is an increase of about $\frac{435}{3001}$ $\cdot$ 100% $\approx$ 14.5% of the smallest value. This means that the model will be slightly more skewed towards the patterns seen in Canberra and Sydney, compared to the rest of the locations, with three exceptions. The last three locations only have 1578 observations. This is about half of the other 46 locations. These three location will therefore have less of an impact in the model training. It is easier for their patterns to be overshadowed by the other locations which have more instances.


In [ ]:
#Plotting the columns into histograms
axes = encoded_eda_dataset.hist(layout= (6,5), figsize=(10,25))
fig = axes.flatten()[0].get_figure()
plt.show()

The features with a normal distribution are: Location, MinTemp, MaxTemp, Temp3pm, Temp9am, WindGustSpeed, Humidity9am, Humidity3pm, Pressure9am, Pressure3pm, WindSpeed9am, WindSpeed3pm

The features with a U-shaped distribution are: Cloud3am, Cloud3pm, Month_Sin, Month_Cos, Day_Sin, Day_Cos, WindGustDir_Cos, WindGustDir_Sin, WindDir9am_Sin, WindDir9am_Cos, WindDir3pm_Sin, WindDir3pm_Cos

The features with a linear distribution are: RainToday, RainTomorrow, Rainfall, Evaporation

Sunshine almost looks like a skewed normal distribution, but the tail end has a lot of instances, and Year almost looks like a uniform distribution, but the lower end has fewer instances than the rest and the upper and has more instances than the rest. 

In [ ]:
sns.kdeplot(encoded_eda_dataset['RainToday'], fill=True, color="blue", label="RainToday")
sns.kdeplot(encoded_eda_dataset['RainTomorrow'], fill=True, color="red", label="RainTomorrow")

plt.title("Overlaying RainToday and RainTomorrow")
plt.xlabel("Values")
plt.ylabel("Density")
plt.legend()
plt.show()

Interestingly the two columns RainToday and RainTomorrow are nearly identical in the kde graph. This is not necessarily a sign of any correlation as it could just mean that any given day has about a 20% chance of rain. 

In [ ]:
#Plotting the boxplots for the dataset
fig, ax = plt.subplots(1,1,figsize=(40, 20))
encoded_eda_dataset.boxplot()

There's not a lot to gain from this as there are too many features for the boxplot, so it makes sense to split it into several plots with similar sizes. 

In [ ]:
pressure_features = ['Pressure9am', 'Pressure3pm']
encoded_eda_dataset[pressure_features].boxplot()

The air pressure is slightly higher in the morning than during the day and this applies to the outliers as well. 

In [ ]:
wind_speed_features = ['WindGustSpeed', 'WindSpeed3pm', 'WindSpeed9am']
encoded_eda_dataset[wind_speed_features].boxplot()

The wind guest speed is higher than the wind speeds as one would expect, but the wind speed is generally higher during the day than during the morning. This is the opposite pattern from what the air pressure showed. WindSpeed9am has a reading that is almost on the same level as the biggest WindGustSpeed outlier, so it might be worth considering for removal. 

In [ ]:
humidity_features = ['Humidity3pm', 'Humidity9am']
encoded_eda_dataset[humidity_features].boxplot()

The humidity is quite a bit higher during the morning than during the day. The humidity at 3pm is more or less evenly spread, although it's mostly concentrated during 

In [ ]:
temp_features = ['MinTemp', 'MaxTemp', 'Temp9am', 'Temp3pm']
encoded_eda_dataset[temp_features].boxplot()

The temperature generally seems to vary by about 10 degrees during the day, but there seems to be more variation in the MaxTemp readings. The highest temperature happens somewhat close to 3pm, but the 9am temperature is quite a bit higher than the lowest temperature. 

In [ ]:
encoded_eda_dataset[['Rainfall']].boxplot()

There are a lot of outliers here. The reason for this could be that it only rains about 20% of days, so 3/4ths of the boxplot will be squished into this 0 rainfall count. The rainfall values exceeding 370 could be candidates for removal as they seem like big outliers.

In [ ]:
#Checking the rainfall without any zeroes
rainfall = encoded_eda_dataset[['Rainfall']]
rainfall_filtered = rainfall.loc[rainfall['Rainfall'] > 0]
rainfall_filtered.boxplot()

Rainfall is mostly concentrated from about one to 20mm of rain, with some outliers reaching all the way up to 370mm. 

In [ ]:
encoded_eda_dataset[['Evaporation']].boxplot()

Evaporation is mostly concentrated around 0-16mm/day, with one outlier reaching all the way up to around 140mm/day. This one could be considered for removal as it's nearly 60 mm/day higher than the second most count. 

In [ ]:
encoded_eda_dataset[['Sunshine']].boxplot()

Sunshine stretches from 0 hours all the way up to over 14 hours of sunshine a day. The average day will have somewhere between 5-10 hours of sunshine. 

In [ ]:
cloud_features = ['Cloud9am', 'Cloud3pm']
encoded_eda_dataset[cloud_features].boxplot()

The lower 50% of instances is more spread out at 9am than 3pm. There are more days with a cloud cover of 0-1 oktas at 9am than there are at 3pm. 

In [ ]:
temp_features = ['Temp9am', 'Temp3pm']
encoded_eda_dataset[temp_features].boxplot()

The temperature is a few degrees higher during the day. The mean temperature at 3pm is about 21 degrees Celsius, while the mean temperature at 9am is about 16 degrees Celsius.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(10,5))

encoded_eda_dataset[['RainToday']].value_counts().plot(kind="bar", ax=axes[0])

axes[0].set_title('RainToday')
axes[0].set_ylabel('Number of Rainy Days')
axes[0].set_xlabel('1 = Yes, 0 = No')
axes[0].tick_params(rotation=0)

encoded_eda_dataset[['RainTomorrow']].value_counts().plot(kind="bar", ax=axes[1])

axes[1].set_title('RainTomorrow')
axes[1].set_ylabel('Number of Rain Predictions')
axes[1].set_xlabel('0 = No, 1 = Yes')
axes[1].tick_params(rotation=0)

There are about 30000 instances of rain in both features and 110000 instances of no rain. 

In [ ]:
encoded_eda_dataset[['Location']].boxplot()

Location has been target encoded, and the probabilities vary from about 5% chance of rain tomorrow to about 35% chance of rain tomorrow. The mean seems to be around 22% which matches with RainTomorrow.  

In [ ]:
eda_dataset[['Date']].hist()

The date feature is close to uniform with 2008 being quite a bit lower than the others. This is most likely due to the records starting late in the year 2008. 


In [ ]:
fig, axes = plt.subplots(1,1, figsize=(15,5))

eda_dataset[['WindGustDir']].value_counts().plot(kind="bar")

axes.set_title('WindGustDir')
axes.set_ylabel('Amount')
axes.set_xlabel('Direction')
axes.tick_params(rotation=0)

The most common wind gust direction is West and the least common is NNE. 

In [ ]:
fig, axes = plt.subplots(1,1, figsize=(15,5))

eda_dataset[['WindDir9am']].value_counts().plot(kind="bar")

axes.set_title('WindDir9am')
axes.set_ylabel('Amount')
axes.set_xlabel('Direction')
axes.tick_params(rotation=0)

The most common wind direction at 9am is north. Interestingly it is much more common for the wind gust to be facing north, than the actual wind direction at 9am. 

In [ ]:
fig, axes = plt.subplots(1,1, figsize=(15,5))

eda_dataset[['WindDir3pm']].value_counts().plot(kind="bar")

axes.set_title('WindDir3pm')
axes.set_ylabel('Amount')
axes.set_xlabel('Direction')
axes.tick_params(rotation=0)

The wind direction at 3pm seems to be very similar to the wind gust direction, although the most common wind direction at 3pm is SE instead of W.  

### Multivariate Analysis

### Time Series Analysis